# Data Enrichment Validation

Validates the enriched dataset against expected rules.

In [ ]:
import pandas as pd
from pathlib import Path

# Load data
csv_path = Path('../../data/Enriched_Data_Final.csv')
df = pd.read_csv(csv_path)
print(f'Loaded {len(df)} rows, {len(df.columns)} columns')

## 1. Column Check
Verify all 28 columns present (12 original + 16 enriched)

In [ ]:
ORIGINAL = ['ID', 'Warehouse_block', 'Mode_of_Shipment', 'Customer_care_calls', 
            'Customer_rating', 'Cost_of_the_Product', 'Prior_purchases', 
            'Product_importance', 'Gender', 'Discount_offered', 'Weight_in_gms', 
            'Reached.on.Time_Y.N']

ENRICHED = ['Customer_type', 'Origin_Region', 'Destination_Region', 'Order_Date',
            'Promised_Date', 'Ship_Date', 'Destination_Arrival_Date', 'Actual_Delivery_Date',
            'Delay_Cause', 'Final_Status', 'Payment_Status', 'Ticket_Raised',
            'Payment_Mode', 'Gap_Shipping_Days', 'Gap_Transit_Days', 'Gap_SLA_Days']

expected = set(ORIGINAL + ENRICHED)
actual = set(df.columns)
missing = expected - actual

print(f'Expected: {len(expected)} columns')
print(f'Actual: {len(actual)} columns')
print(f'Missing: {missing if missing else "None ✅"}')

## 2. NULL Rates
~18% Actual_Delivery_Date NULL (undelivered orders)

In [ ]:
null_rate = df['Actual_Delivery_Date'].isna().mean() * 100
status = '✅' if 10 <= null_rate <= 25 else '❌'
print(f'Actual_Delivery_Date NULL: {null_rate:.1f}% {status} (expected ~18%)')

## 3. Date Sequence
Order < Ship < Arrival < Delivery

In [ ]:
date_cols = ['Order_Date', 'Ship_Date', 'Destination_Arrival_Date', 'Actual_Delivery_Date']
df_dates = df[date_cols].copy()
for col in date_cols:
    df_dates[col] = pd.to_datetime(df_dates[col], errors='coerce')

valid = df_dates.dropna()
violations = (
    (valid['Order_Date'] > valid['Ship_Date']).sum() +
    (valid['Ship_Date'] > valid['Destination_Arrival_Date']).sum() +
    (valid['Destination_Arrival_Date'] > valid['Actual_Delivery_Date']).sum()
)
status = '✅' if violations == 0 else '❌'
print(f'Date sequence violations: {violations} {status} (checked {len(valid)} rows)')

## 4. Region Values
Expected: North, South, East, West, Midwest

In [ ]:
EXPECTED_REGIONS = {'North', 'South', 'East', 'West', 'Midwest'}
actual_regions = set(df['Origin_Region'].unique()) | set(df['Destination_Region'].unique())
invalid = actual_regions - EXPECTED_REGIONS
status = '✅' if not invalid else '❌'
print(f'Regions: {actual_regions}')
print(f'Invalid: {invalid if invalid else "None"} {status}')

## 5. Customer Types
Expected: Consumer, Corporate, Home Office

In [ ]:
dist = df['Customer_type'].value_counts(normalize=True) * 100
print('Customer Type Distribution:')
print(dist.round(1))

## 6. Payment Status
Expected: Paid ~78%, Refunded ~18%, COD ~4%

In [ ]:
dist = df['Payment_Status'].value_counts(normalize=True) * 100
print('Payment Status Distribution:')
print(dist.round(1))

## 7. Ticket Raised
Expected: 0/1 values, ~41% raised

In [ ]:
ticket_rate = (df['Ticket_Raised'] == 1).mean() * 100
print(f'Ticket Raised: {ticket_rate:.1f}% ✅')

## Summary

In [ ]:
print('='*50)
print('VALIDATION COMPLETE')
print('='*50)